In [ ]:

# XGBOOST 


import numpy as np
import pandas as pd
from xgboost import XGBRanker
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
TARGET = "RET"

# DATA PREPARATION FUNCTION

def prepare_rank_data(df, features):
    X = df[features].fillna(0)
    y = df[TARGET].values

    # group = number of samples per DATE
    groups = df.groupby("DATE").size().values
    return X, y, groups

# CROSS-VALIDATION FUNCTION

def cross_val_xgb_rank(train_df, features, n_splits=5):

    dates = train_df["DATE"].unique()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    scores = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(dates)):
        d_tr = dates[tr_idx]
        d_va = dates[va_idx]

        df_tr = train_df[train_df["DATE"].isin(d_tr)]
        df_va = train_df[train_df["DATE"].isin(d_va)]

        X_tr, y_tr, g_tr = prepare_rank_data(df_tr, features)
        X_va, y_va, g_va = prepare_rank_data(df_va, features)

        model = XGBRanker(
            n_estimators=1500,
            learning_rate=0.03,
            max_depth=6,
            min_child_weight=50,
            subsample=0.7,
            colsample_bytree=0.7,
            objective="rank:pairwise",
            eval_metric="ndcg",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        model.fit(
            X_tr, y_tr,
            group=g_tr,
            eval_set=[(X_va, y_va)],
            eval_group=[g_va],
            verbose=False,
            early_stopping_rounds=100
        )

        # Post-processing (median par DATE)
        df_va = df_va.copy()
        df_va["score"] = model.predict(X_va)

        y_pred = df_va.groupby("DATE")["score"] \
                       .transform(lambda x: x > x.median()) \
                       .astype(int)

        acc = accuracy_score(y_va, y_pred)
        scores.append(acc)

        print(f"Fold {fold+1}: {acc:.4f}")

    print(f"\nCV mean: {np.mean(scores):.4f}")
    return np.mean(scores)
